# 03 — Air Quality Interpolation (PM2.5)

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | OpenAQ PM2.5 stations (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · Random Forest · GBM · Regression Kriging |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live OpenAQ data (requires API key).  
**🗺 Interactive maps** use [geemap](https://geemap.org).

In [ ]:
%pip install -q "geointerpo[full]" geemap localtileserver

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY      = (-120.0, 49.0, -110.0, 60.0)  # Alberta, Canada  (min_lon, min_lat, max_lon, max_lat)
PLACE         = 'Alberta, Canada'              # 🌐 resolves to actual province boundary via Nominatim
DATE          = '2024-01-15'
RESOLUTION    = 0.5   # degrees (~50 km)
OPENAQ_API_KEY = ''   # Get a free key at https://explore.openaq.org/register

---
## Part A — Offline demo (no network needed)

In [ ]:
result = Pipeline(
    data='sample',                 # Step 1 — built-in synthetic PM2.5 data
    variable='air_quality',
    boundary=BOUNDARY,             # Step 2 — four corners
    method=['idw', 'kriging', 'rf', 'gbm'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'exponential'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
result.metrics_table()

In [ ]:
result.plot(cmap='YlOrRd')
plt.suptitle('PM2.5 — Alberta, Canada (synthetic demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='YlOrRd', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('PM2.5 station observations — Alberta (synthetic, µg/m³)')
plt.tight_layout()

### City boundary clipping

Restrict the output to a specific city boundary — bbox is derived automatically.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, city in zip(axes, ['Calgary, Alberta', 'Edmonton, Alberta']):
    city_result = Pipeline(
        data='sample',
        variable='air_quality',
        boundary=city,
        method=['idw', 'kriging'],
        resolution=0.05,
        cv_folds=3,
    ).run()
    city_result.grid.plot(ax=ax, cmap='YlOrRd')
    city_result.stations.plot(ax=ax, color='k', markersize=20, zorder=5)
    ax.set_title(city.split(',')[0])

plt.suptitle('PM2.5 — Calgary & Edmonton (synthetic, µg/m³)', y=1.02)
plt.tight_layout()
plt.show()

---
## Part B — 🌐 Live OpenAQ data

Fetches real PM2.5 station data from OpenAQ v3.

**Setup:** Set `OPENAQ_API_KEY` in the Configuration cell above.  
Get a free key at [explore.openaq.org/register](https://explore.openaq.org/register).

In [ ]:
if not OPENAQ_API_KEY:
    print("⚠️  Set OPENAQ_API_KEY in the Configuration cell to run Part B.")
    print("   Get a free key at https://explore.openaq.org/register")
    result_live = None
else:
    result_live = Pipeline(
        data='openaq',
        variable='pm25',
        date=DATE,
        boundary=BOUNDARY,
        method=['idw', 'kriging', 'rf', 'gbm'],
        method_params={
            'kriging': {'variogram_model': 'exponential'},
        },
        resolution=RESOLUTION,
        cv_folds=5,
        openaq_api_key=OPENAQ_API_KEY,
    ).run()
    print(f'Loaded {len(result_live.stations)} OpenAQ stations')

In [ ]:
if result_live:
    result_live.metrics_table()

In [ ]:
if result_live:
    result_live.plot(cmap='YlOrRd')
    plt.suptitle(f'PM2.5 — Alberta {DATE}  (OpenAQ)', y=1.02)
    plt.show()

### Regression Kriging with elevation covariate

Uses SRTM elevation as a covariate — topography can influence PM2.5 dispersion.

In [ ]:
result_rk = Pipeline(
    data='sample',
    variable='air_quality',
    boundary=BOUNDARY,
    method=['kriging', 'rk'],
    method_params={'kriging': {'variogram_model': 'exponential'}},
    include_dem=True,
    dem_source='synthetic',        # use 'gee' or 'srtm' with network
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print('Ordinary Kriging vs Regression Kriging (with DEM):')
result_rk.metrics_table()

### Variogram estimation with scikit-gstat

[scikit-gstat](https://github.com/mmaelicke/scikit-gstat) provides rich variogram analysis with six semi-variance estimators (Matheron, Cressie, Dowd, Genton, entropy…) and six model functions. Here we estimate the experimental variogram of the synthetic PM2.5 field directly from station coordinates and compare fitted parameters — **nugget** (micro-scale variance), **sill** (total variance), and **range** (correlation length) — across spherical, exponential, and Matérn models.

In [ ]:
%pip install -q scikit-gstat

In [ ]:
import skgstat as skg
import numpy as np
import pandas as pd

coords = np.column_stack([result.stations.geometry.x, result.stations.geometry.y])
values = result.stations['value'].values

rows = []
for model in ['spherical', 'exponential', 'gaussian', 'matern']:
    V = skg.Variogram(coordinates=coords, values=values, model=model, n_lags=10)
    p = V.parameters  # [range, sill, nugget]
    rows.append({
        'model':  model,
        'range':  round(p[0], 4),
        'sill':   round(p[1], 3),
        'nugget': round(p[2], 3),
    })

print('Experimental variogram fits — PM2.5 station data (scikit-gstat):')
pd.DataFrame(rows).set_index('model')

In [ ]:
for model in ['spherical', 'exponential']:
    V = skg.Variogram(coordinates=coords, values=values, model=model, n_lags=10)
    fig = V.plot()
    fig.suptitle(f'Variogram — {model} model (PM2.5 Alberta, scikit-gstat)', y=1.02)
    plt.show()

### All-methods comparison — Alberta

Run all supported methods and compare them visually and by CV metrics.

In [ ]:
result_all = Pipeline(
    data='sample',
    variable='air_quality',
    boundary=BOUNDARY,
    method=['idw', 'kriging', 'rbf', 'rf', 'gbm', 'natural_neighbor'],
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'exponential'},
        'rbf':    {'kernel': 'thin_plate_spline'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print('All-methods CV metrics:')
result_all.metrics_table()

In [ ]:
result_all.plot(cmap='YlOrRd')
plt.suptitle('PM2.5 — All methods comparison — Alberta (synthetic, µg/m³)', y=1.02)
plt.show()

### Interactive map

In [ ]:
import geemap, tempfile

with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as f:
    tmp = f.name
result.grid.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326").rio.to_raster(tmp)

m = geemap.Map(center=[54.5, -115.0], zoom=6, ee_initialize=False)
m.add_raster(tmp, colormap="ylorrd", layer_name="PM2.5")
m.add_layer_manager()
m

---
## Save outputs

In [ ]:
result.save('outputs/air_quality', geotiff=True, plot=True)
print('Saved to outputs/air_quality/')